In [1]:
import torch
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2470, 0.2435, 0.2616])
])
train_set = datasets.CIFAR10(
    download = True,
    root = 'data',
    transform = transform,
    train = True
)
test_set = datasets.CIFAR10(
    download = True,
    train = False,
    root = 'data',
    transform = transform
)
trainloader = DataLoader(
    train_set,
    shuffle = True,
    batch_size = 256
)
testloader = DataLoader(
    test_set,
    shuffle = True,
    batch_size = 256
)

In [6]:
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super(CNNModel, self).__init__()
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(32)
        )
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(64)
        )
        self.conv_layer4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer5 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        self.conv_layer6 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        self.conv_layer7 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2)
        )

        self.conv_layer8 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding = 'same', stride = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )
        self.conv_layer9 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding = 'same', stride = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )
        self.conv_layer10 = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
        self._init_weights()
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = self.conv_layer5(x)
        x = self.conv_layer6(x)
        x = self.conv_layer7(x)
        x = self.conv_layer8(x)
        x = self.conv_layer9(x)
        x = self.conv_layer10(x)
        out = self.classifier(x)

        return out 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode = 'fan_in',
                                        nonlinearity = 'relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        

In [7]:
model = CNNModel(10)
model.to(device)
print(model)

CNNModel(
  (conv_layer1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer2): Sequential(
    (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer4): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): MaxPool2d(kern

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 1e-3)


In [9]:
def evaluate(model, criterion, testloader):
    model.eval()
    test_loss = 0.0
    running_correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            predicted = torch.argmax(outputs.data, dim = 1)
            total += labels.size(0)
            running_correct += (predicted == labels).sum().item()
    accuracies = running_correct * 100 / total
    test_loss = test_loss / len(testloader)
    return test_loss, accuracies

In [ ]:
epochs = 100
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0
    for X, y in trainloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)

        loss = criterion(outputs, y)
        running_loss += loss.item()
        loss.backward()
        optimizer.step()

        predicted = torch.argmax(outputs.data, dim = 1)
        total += y.size(0)
        running_correct += (predicted == y).sum().item()
    epoch_accuracy = running_correct * 100 / total
    epoch_loss = running_loss / total

    test_loss, test_accuracy = evaluate(model, criterion, testloader)
    print(f'Epoch [{epochs + 1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy : {epoch_accuracy:.2f}%, Test Loss: {test_loss:.4f}, Test Accuracy : {test_accuracies:.2f}'%)
